# 01 — Monthly membership analysis

Reconstruct monthly membership counts for January 2015–December 2018. Earlier
registrations contribute to the opening balance. Customer intervals run from
first registration to recorded final exit or the observation cutoff.


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/raw.xlsx").is_file()
             and (p / "src/analysis.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.pipeline import load_data, preparation_tables, monthly_tables, save_tables

raw, prepared, members = load_data()
OUT = ROOT / "results"


For each month:
- New enrolments are first registrations dated in that month.
- Dropouts are final exits with `Dropout = 1` dated in that month.
- Closing membership equals opening membership + enrolments − dropouts.
- Dropout rate is exits among opening members / opening membership.

An exit is removed on its recorded date. Records open at the observation cutoff
are retained. Formulas are in `monthly` and `monthly_diagnostics` in `src/`.


In [2]:
tables = monthly_tables(members)
tables["MonthlyData"][["MonthStart", "OpeningMemberships", "NewEnrolments",
                       "Dropouts", "ActiveMemberships", "DropoutRate"]].head(12)

,MonthStart,OpeningMemberships,NewEnrolments,Dropouts,ActiveMemberships,DropoutRate
0,2015-01-01,2001,219,1,2219,0.000500
1,2015-02-01,2219,269,0,2488,0.000000
2,2015-03-01,2488,539,3,3024,0.001206
3,2015-04-01,3024,309,2,3331,0.000661
4,2015-05-01,3331,156,1,3486,0.000300
5,2015-06-01,3486,104,0,3590,0.000000
6,2015-07-01,3590,126,50,3666,0.013649
7,2015-08-01,3666,72,1,3737,0.000273
8,2015-09-01,3737,651,1856,2532,0.496655
9,2015-10-01,2532,621,13,3140,0.005134


Annual new enrolments and dropouts are sums of monthly events.
`ClosingMemberships` is the December closing stock, not a cancellation count.


In [3]:
tables["AnnualSummary"][["Year", "OpeningMemberships", "NewEnrolments",
                         "Dropouts", "ClosingMemberships", "NetMembershipChange"]]

,Year,OpeningMemberships,NewEnrolments,Dropouts,ClosingMemberships,NetMembershipChange
0,2015,2001,3451,2154,3298,1297
1,2016,3298,2921,2746,3473,175
2,2017,3473,2314,1882,3905,432
3,2018,3905,2521,2599,3827,-78


`AnnualEnrolmentShare` divides a month's first registrations by that
year's total. The seasonal summary averages each calendar month's four annual
values with equal weight per year. Rates in the saved CSV files are fractions.


In [4]:
tables["SeasonalSummary"][["MonthName", "MeanNewEnrolments",
                            "MeanAnnualEnrolmentShare", "MeanDropoutRate"]]

,MonthName,MeanNewEnrolments,MeanAnnualEnrolmentShare,MeanDropoutRate
0,January,186.25,0.066982,0.034968
1,February,236.00,0.084081,0.027200
2,March,391.00,0.138197,0.024831
3,April,254.00,0.089750,0.026692
4,May,137.25,0.049204,0.033724
5,June,86.25,0.031032,0.043562
6,July,101.75,0.036477,0.166833
7,August,68.50,0.024710,0.046802
8,September,551.75,0.197460,0.134112
9,October,482.75,0.172811,0.012639


Inspect the largest exit dates by year alongside the seasonal
average. This identifies peaks concentrated in a particular year.


In [5]:
tables["LargestExitDates"]

,Year,FinishDate,DropoutsOnDate,ShareOfAnnualDropouts
0,2015,2015-09-16,1682,0.780873
1,2015,2015-09-14,146,0.067781
2,2016,2016-07-31,890,0.324108
3,2016,2016-07-08,281,0.102331
4,2017,2017-08-29,534,0.283741
5,2017,2017-07-31,147,0.078108
6,2018,2018-07-31,799,0.307426
7,2018,2018-07-08,307,0.118122


In [6]:
save_tables(OUT / "monthly", tables)
tables["MonthlyReconciliation"].drop(columns="MonthStart").abs().max().rename("Maximum residual")

StockFlowResidual            0.0
DropoutPartitionResidual     0.0
IndependentLedgerResidual    0.0
RateResidual                 0.0
Name: Maximum residual, dtype: float64